# Financial data collection from yFinance

In [1]:
# !pip install yfinance
# !pip install pandas
import yfinance as yf
import pandas as pd
import numpy as np
from datetime import datetime
import pprint

def fetch_financials_from_yfinance(ticker, fiscal_year_end_dates=None):
    """
    Fetches comprehensive financial data for a ticker.
    Returns a list of dictionaries, one per fiscal year, with all metrics.

    Parameters:
        ticker (str): Stock ticker symbol.
        fiscal_year_end_dates (list of dates, optional): If provided, will attempt to fetch market cap on those dates.
            Otherwise, uses current market cap.

    Returns:
        list of dict: Each dict contains metrics for one fiscal year.
    """
    try:
        stock = yf.Ticker(ticker)

        # ----- 1. Get financial statements -----
        income = stock.financials
        balance = stock.balance_sheet
        cashflow = stock.cashflow

        if income.empty:
            return []

        # ----- 2. Get company info (sector, industry) -----
        info = stock.info
        sector = info.get('sector')
        industry = info.get('industry')

        # ----- 3. Get historical market cap (optional) -----
        # We need historical price data to compute EV at fiscal year end.
        # This is expensive; if not provided, we use current market cap.
        hist_price = None
        if fiscal_year_end_dates:
            # Download daily prices for the last few years
            hist_price = stock.history(period="5y")

        # ----- 4. Process each fiscal year -----
        results = []
        # Income statement columns are dates (fiscal year ends)
        for col in income.columns:
            year = col.year
            # Skip if we don't have balance sheet or cash flow for same column
            if col not in balance.columns or col not in cashflow.columns:
                continue

            # --- Income statement items ---
            revenue = income.loc['Total Revenue'].get(col) if 'Total Revenue' in income.index else income.loc['Revenue'].get(col) if 'Revenue' in income.index else None
            ebitda = income.loc['EBITDA'].get(col) if 'EBITDA' in income.index else None
            ebit = income.loc['EBIT'].get(col) if 'EBIT' in income.index else income.loc['Operating Income'].get(col) if 'Operating Income' in income.index else None
            net_income = income.loc['Net Income'].get(col) if 'Net Income' in income.index else None

            # --- Balance sheet items ---
            total_assets = balance.loc['Total Assets'].get(col) if 'Total Assets' in balance.index else None
            total_liabilities = balance.loc['Total Liabilities'].get(col) if 'Total Liabilities' in balance.index else None
            shareholders_equity = balance.loc['Stockholders Equity'].get(col) if 'Stockholders Equity' in balance.index else None
            current_assets = balance.loc['Current Assets'].get(col) if 'Current Assets' in balance.index else None
            current_liabilities = balance.loc['Current Liabilities'].get(col) if 'Current Liabilities' in balance.index else None
            short_term_debt = balance.loc['Short Term Debt'].get(col) if 'Short Term Debt' in balance.index else 0
            long_term_debt = balance.loc['Long Term Debt'].get(col) if 'Long Term Debt' in balance.index else 0
            cash_equiv = balance.loc['Cash And Cash Equivalents'].get(col) if 'Cash And Cash Equivalents' in balance.index else 0

            total_debt = short_term_debt + long_term_debt

            # --- Cash flow items ---
            operating_cf = cashflow.loc['Operating Cash Flow'].get(col) if 'Operating Cash Flow' in cashflow.index else None
            capex = cashflow.loc['Capital Expenditure'].get(col) if 'Capital Expenditure' in cashflow.index else None
            fcf = (operating_cf + capex) if operating_cf is not None and capex is not None else None

            # --- Market cap at fiscal year end ---
            if hist_price is not None and col in hist_price.index:
                # Use the closing price on the fiscal year end date (or nearest available)
                try:
                    price = hist_price.loc[col.strftime('%Y-%m-%d')]['Close'] if col in hist_price.index else hist_price.asof(col)['Close']
                    shares_outstanding = info.get('sharesOutstanding')
                    if shares_outstanding and price:
                        market_cap = price * shares_outstanding
                    else:
                        market_cap = None
                except:
                    market_cap = None
            else:
                # Fallback: use current market cap
                market_cap = info.get('marketCap')

            # --- Enterprise Value (EV) ---
            ev = None
            if market_cap is not None and total_debt is not None and cash_equiv is not None:
                ev = market_cap + total_debt - cash_equiv

            # --- Ratios ---
            # EBITDA margin
            ebitda_margin = (ebitda / revenue) if ebitda and revenue and revenue != 0 else None
            # FCF margin
            fcf_margin = (fcf / revenue) if fcf and revenue and revenue != 0 else None
            # Debt/EBITDA
            debt_to_ebitda = (total_debt / ebitda) if total_debt and ebitda and ebitda != 0 else None
            # Interest coverage (EBIT / Interest Expense)
            interest_expense = income.loc['Interest Expense'].get(col) if 'Interest Expense' in income.index else None
            interest_coverage = (ebit / interest_expense) if ebit and interest_expense and interest_expense != 0 else None
            # Current ratio
            current_ratio = (current_assets / current_liabilities) if current_assets and current_liabilities and current_liabilities != 0 else None
            # ROA, ROE
            roa = (net_income / total_assets) if net_income and total_assets and total_assets != 0 else None
            roe = (net_income / shareholders_equity) if net_income and shareholders_equity and shareholders_equity != 0 else None

            # --- Growth metrics (needs previous year) ---
            # We'll compute year-over-year changes after collecting all years

            # Assemble dictionary
            metrics = {
                'ticker': ticker,
                'fiscal_year_end': col,
                'sector': sector,
                'industry': industry,
                'revenue': revenue,
                'ebitda': ebitda,
                'ebitda_margin': ebitda_margin,
                'net_income': net_income,
                'total_assets': total_assets,
                'total_liabilities': total_liabilities,
                'shareholders_equity': shareholders_equity,
                'total_debt': total_debt,
                'cash_equiv': cash_equiv,
                'operating_cf': operating_cf,
                'capex': capex,
                'fcf': fcf,
                'fcf_margin': fcf_margin,
                'debt_to_ebitda': debt_to_ebitda,
                'interest_coverage': interest_coverage,
                'current_ratio': current_ratio,
                'roa': roa,
                'roe': roe,
                'market_cap': market_cap,
                'enterprise_value': ev,
                # Valuation multiples
                'ev_to_ebitda': (ev / ebitda) if ev and ebitda and ebitda != 0 else None,
                'price_to_sales': (market_cap / revenue) if market_cap and revenue and revenue != 0 else None
            }
            results.append(metrics)

        # Add year-over-year growth columns
        results = add_growth_metrics(results)
        results_dict = {}
        for metrics in results:
            year = metrics['year']
            results_dict[year] = metrics   # store whole metrics dict per year
        return results_dict
        # return results

    except Exception as e:
        print(f"Error fetching financials for {ticker}: {e}")
        return []

def add_growth_metrics(results):
    """Adds YoY growth rates for revenue, EBITDA, etc."""
    if len(results) < 2:
        return results
    # Sort by year ascending
    results.sort(key=lambda x: x['year'])
    for i in range(1, len(results)):
        prev = results[i-1]
        curr = results[i]
        # Revenue growth
        if prev['revenue'] and curr['revenue'] and prev['revenue'] != 0:
            curr['revenue_growth'] = (curr['revenue'] - prev['revenue']) / abs(prev['revenue'])
        else:
            curr['revenue_growth'] = None
        # EBITDA growth
        if prev['ebitda'] and curr['ebitda'] and prev['ebitda'] != 0:
            curr['ebitda_growth'] = (curr['ebitda'] - prev['ebitda']) / abs(prev['ebitda'])
        else:
            curr['ebitda_growth'] = None
        # FCF growth
        if prev['fcf'] and curr['fcf'] and prev['fcf'] != 0:
            curr['fcf_growth'] = (curr['fcf'] - prev['fcf']) / abs(prev['fcf'])
        else:
            curr['fcf_growth'] = None
    return results
    
msft_df = fetch_financials_from_yfinance("MSFT")
pprint.pprint(msft_df, indent=4)

Error fetching financials for MSFT: 'year'
[]


# Text data from SEC 10-K filling

In [2]:
# !pip install edgartools
import re
from bs4 import BeautifulSoup
from edgar import *
import requests


set_identity("oscarchanly@gmail.com")

headers = {'User-Agent':'Dummy Company oscarchanly@gmail.com','Accept-Encoding':'gzip, deflate','Host':'www.sec.gov'}


def get_n_year_10K(ticker, n = 5): 

    """
    Extract 10 K filings in the last n years
    """

    # get 10-K filing from last n year 
    company = Company(ticker)

    filings_5y = company.get_filings(form="10-K").latest(n) 

   
    filings_list = [filing for filing in filings_5y]

    # print summary
    for filing in filings_list:
        print(f"{filing.filing_date}: {filing.form} - {filing.company}")

    return filings_list

def normalize_text(text):
    """

    clean the text for pattern matching

    """
    # Replace non-breaking spaces
    text = text.replace('\u00A0', ' ')
    # # Replace curly apostrophe and other quotes with ASCII apostrophe
    text = text.replace('’', "'").replace('‘', "'").replace('`', "'")

    return text

def extract_mda_from_filing(filing):

    """
    Extract the MD&A (Item 7) text from a 10-K Filing object.
    """

    # Get the primary document's HTML
    url = filing.filing_url
    response = requests.get(url, headers = headers)
    html_content = response.text
    soup = BeautifulSoup(html_content, 'html.parser')
    text = soup.get_text(separator='\n')

    start_patterns = [
        r"item[ \t]*7\.?[ \t]*management's",
        r"item[ \t]*7\.?[ \t]*–[ \t]*management's",
    ]
    end_patterns = [
        r"item[ \t]*7a\.?[ \t]*quantitative",
        r"item[ \t]*8\.?[ \t]*financial\s+statements",
        r"item[ \t]*8\.?[ \t]*–[ \t]*financial\s+statements"
    ]
    lower_text = normalize_text(text.lower())
    
    start_idx = None
    for pat in start_patterns:
        match = re.search(pat, lower_text)
        if match:
            start_idx = match.start()
            break
    if start_idx is None:
        return None
    
    end_idx = len(text)
    for pat in end_patterns:
        match = re.search(pat, lower_text[start_idx:])
        if match:
            end_idx = start_idx + match.start()
            break
        
    
    mda_text = text[start_idx:end_idx].strip()
    mda_text = re.sub(r'^item\s*7\.?\s*', '', mda_text, flags=re.IGNORECASE).strip()
    print(url)
    return mda_text

# demostration 

filing_list_5y = get_n_year_10K("AAPL")

print(extract_mda_from_filing(filing_list_5y[0]))

C:\Users\Oscar\AppData\Roaming\Python\Python314\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2025-10-31: 10-K - Apple Inc.
2024-11-01: 10-K - Apple Inc.
2023-11-03: 10-K - Apple Inc.
2022-10-28: 10-K - Apple Inc.
2021-10-29: 10-K - Apple Inc.
https://www.sec.gov/Archives/edgar/data/320193/000032019325000079/aapl-20250927.htm
Management’s Discussion and Analysis of Financial Condition and Results of Operations
The following discussion should be read in conjunction with the consolidated financial statements and accompanying notes included in Part II, Item 8 of this Form 10-K. This Item generally discusses 2025 and 2024 items and year-to-year comparisons between 2025 and 2024. Discussions of 2023 items and year-to-year comparisons between 2024 and 2023 are not included, and can be found in “Management’s Discussion and Analysis of Financial Condition and Results of Operations” in Part II, Item 7 of the Company’s Annual Report on Form 10-K for the fiscal year ended September 28, 2024.
Product, Service and Software Announcements
The Company announces new product, service and softwar

## Stagnation measurement from text data

In [3]:
filing_list_5y = get_n_year_10K("AAPL")
mda_by_year = {}
for filing in filing_list_5y:
    year = filing.filing_date.year
    mda_text = extract_mda_from_filing(filing)
    if mda_text:
        mda_by_year[year] = mda_text

2025-10-31: 10-K - Apple Inc.
2024-11-01: 10-K - Apple Inc.
2023-11-03: 10-K - Apple Inc.
2022-10-28: 10-K - Apple Inc.
2021-10-29: 10-K - Apple Inc.
https://www.sec.gov/Archives/edgar/data/320193/000032019325000079/aapl-20250927.htm
https://www.sec.gov/Archives/edgar/data/320193/000032019324000123/aapl-20240928.htm
https://www.sec.gov/Archives/edgar/data/320193/000032019323000106/aapl-20230930.htm
https://www.sec.gov/Archives/edgar/data/320193/000032019322000108/aapl-20220924.htm
https://www.sec.gov/Archives/edgar/data/320193/000032019321000105/aapl-20210925.htm


In [4]:
import re
import numpy as np
import pandas as pd
from collections import Counter
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
from scipy.stats import pearsonr


# ------------------------------------------------------------
# 1. Innovation Vocabulary Decay
# ------------------------------------------------------------
forward_words = {
    'anticipate', 'expect', 'believe', 'future', 'plan', 'potential',
    'opportunity', 'goal', 'objective', 'strategy', 'forecast', 'project',
    'outlook', 'target', 'aim', 'seek', 'hope', 'intend', 'look forward'
}

def compute_innovation_decay(mda_by_year):
    """
    Returns:
        freq: dict year -> forward-looking word frequency
        slope: linear slope over years (negative = decay)
    """
    years = sorted(mda_by_year.keys())
    freq = {}
    for year in years:
        text = mda_by_year[year]
        words = re.findall(r'\b[a-zA-Z]+\b', text.lower())
        total = len(words)
        if total == 0:
            freq[year] = 0
            continue
        fwd_count = sum(1 for w in words if w in forward_words)
        freq[year] = fwd_count / total
    # Compute trend
    if len(years) >= 3:
        x = np.array(years)
        y = np.array([freq[y] for y in years])
        slope = np.polyfit(x, y, 1)[0]
    else:
        slope = None
    return freq, slope
compute_innovation_decay(mda_by_year)

({2021: 0.003067484662576687,
  2022: 0.0016556291390728477,
  2023: 0.001405152224824356,
  2024: 0.001440922190201729,
  2025: 0.002041649652919559},
 np.float64(-0.00022663769681853646))

In [5]:
# ------------------------------------------------------------
# 2. Topic Rigidity Score (using sentence embeddings)
# ------------------------------------------------------------
# Load a sentence transformer model once
embed_model = SentenceTransformer('all-MiniLM-L6-v2')

def compute_topic_rigidity(mda_by_year):
    """
    Returns:
        rigidity: average cosine similarity between consecutive years
        similarities: list of yearly similarities
    """
    years = sorted(mda_by_year.keys())
    if len(years) < 2:
        return None, []
    texts = [mda_by_year[y] for y in years]
    # Encode all texts
    embeddings = embed_model.encode(texts)
    similarities = []
    for i in range(len(years)-1):
        sim = cosine_similarity([embeddings[i]], [embeddings[i+1]])[0][0]
        similarities.append(sim)
    rigidity = np.mean(similarities) if similarities else None
    return rigidity, similarities

compute_topic_rigidity(mda_by_year)

(np.float32(0.78605175),
 [np.float32(0.6875325),
  np.float32(0.7674374),
  np.float32(0.71229494),
  np.float32(0.976942)])

In [6]:
# ------------------------------------------------------------
# 3. Strategic Topic Decay
# ------------------------------------------------------------
strategic_keywords = {
    'innovation', 'research', 'development', 'rd', 'new product',
    'launch', 'expansion', 'market share', 'growth', 'opportunity',
    'investment', 'initiative', 'breakthrough', 'pipeline', 'strategic'
}

def compute_strategic_decay(mda_by_year):
    """
    Returns:
        freq: dict year -> strategic keyword frequency
        slope: linear slope (negative = decay)
    """
    years = sorted(mda_by_year.keys())
    freq = {}
    for year in years:
        text = mda_by_year[year]
        words = re.findall(r'\b[a-zA-Z]+\b', text.lower())
        total = len(words)
        if total == 0:
            freq[year] = 0
            continue
        # Count occurrences of any strategic keyword (simple word match)
        strat_count = sum(1 for w in words if w in strategic_keywords)
        freq[year] = strat_count / total
    if len(years) >= 3:
        x = np.array(years)
        y = np.array([freq[y] for y in years])
        slope = np.polyfit(x, y, 1)[0]
    else:
        slope = None
    return freq, slope


compute_strategic_decay(mda_by_year)

({2021: 0.003834355828220859,
  2022: 0.0024834437086092716,
  2023: 0.00234192037470726,
  2024: 0.0024015369836695487,
  2025: 0.0032666394446712946},
 np.float64(-0.00012173394920389689))